In [ ]:
# Cell 1: Environment Setup

include("../scripts/dictionaries.jl")
include("../scripts/helpers.jl")
using .WorkflowHelpers
using OMJulia

# --- Configuration ---

# 1. Directory containing the single-file model
MODEL_DIR = abspath("models")

# 2. Select the model to build
MODEL = "MyBESS"

# 3. Path to the selected model file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Your Dynawo installation (used only for its Modelica Standard Library)
DYNAWO_DIR = "/home/clarafercas/dynawo"
MODELICA_PKG_PATH = "$DYNAWO_DIR/OpenModelica/lib/omlibrary/Modelica/package.mo"

# 5. Dynawo Modelica library from this repo
DYNAWO_PKG_PATH = abspath("../dynawo_library/Dynawo/package.mo")

# 6. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = Dict{String, String}(
    # "generatorSynchronous" => "GeneratorSynchronousInt_INIT",
)

# 7. Slack component (leave empty to disable slack-specific handling).
SLACK_COMPONENT = ""

In [ ]:
# Cell 2: OpenModelica Setup + Single Model Validation

# 1. Start OMC and load libraries
omc = OMJulia.OMCSession()
omc_call(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(omc, "loadModel(Complex)")
omc_call(omc, "loadModel(ModelicaServices)")
omc_call(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# 2. Load the selected model and validate the configuration against it
omc_call(omc, "loadFile(\"$MODEL_FILE_PATH\")")
check_user_configuration_single(omc;
    model = MODEL,
    slack_component = SLACK_COMPONENT,
    init_model_by_component = INIT_MODEL_BY_COMPONENT,
)

In [ ]:
# Cell 3: Auxiliary Model Setup
OUTPUT_DIR = abspath("outputs")
mkpath(OUTPUT_DIR)

AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE = joinpath(OUTPUT_DIR, AUX_MODEL * ".mo")

In [ ]:
# Cell 4: Single-Model Auxiliary Build Pipeline

# Create/refresh the auxiliary model in OpenModelica
sendExpression(omc, "deleteClass($AUX_MODEL)")
omc_call(omc, "copyClass($MODEL, \"$AUX_MODEL\")")

# Build component dictionary from the source model
components = get_all_components(omc, MODEL)

# Apply dictionary-driven replacements
apply_replacements!(omc, MODEL, AUX_MODEL, components, SLACK_COMPONENT)

# Delete connections to cleanup targets
delete_connections!(omc, AUX_MODEL, components)

# Delete cleanup-target components
delete_components!(omc, AUX_MODEL, components)

# Add INIT models for the source model
add_init_models!(omc, MODEL, AUX_MODEL, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Add load-flow modifiers
apply_LF_modifiers!(omc, MODEL, AUX_MODEL, components)

# Add initial equations for the source model
add_init_equations!(omc, MODEL, AUX_MODEL, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Clean up the auxiliary model equations in place
clean_aux_equations!(omc, AUX_MODEL, components, SLACK_COMPONENT)

# Validate the build
chk = sendExpression(omc, "checkModel($AUX_MODEL)", parsed=false)
println(chk)

# Save the auxiliary model
omc_call(omc, "saveModel(\"$AUX_FILE\", $AUX_MODEL)")